# MU 매출·영업이익 나우캐스트 분석 v7
**v7 변경점**: Cell 9 의 NameError(NOWCAST_FQ 미정의) 수정 - 대상 분기를 패널 마지막 실적 분기 + 3개월로 자동 도출 (Cell 8 의 하드코딩 월도 제거, 분기마다 수동 갱신 불필요). 분기 중간 실행 시 데이터 부족 경고 추가.

**v6 변경점**: 난야 실적 국면 판별 신호(Cell 12: 3M-MA YoY 의 ΔYoY, 주가 백테스트 v2 채택 신호) + 분기 카드 x 월간 국면 종합 판정(Cell 13) 추가. 검증 근거: MU_price_signal_backtest_v2 (Sharpe 1.45, MDD -26%, 전환 6회/75개월).

**v5 변경점**: FMP 컨센서스 수집(Cell 9) + 매수/매도 신호 판정(Cell 10) + 컨센서스 vintage 스냅샷 적재(Cell 11) 추가. 응답 필드명은 stable/legacy 어느 쪽이든 자동 인식하므로 사전 테스트 불필요.

**v4 변경점**: MU 재무 데이터 경로 확정 (`미국주식\raw_data\MU_fs_data.csv`).

**v3 변경점**: Control Panel에 실제 데이터 경로 반영 — 수출은 `한국_월간수출분석\{DATA_YM}\` 폴더, 대만은 `지표상회_복제\...\tw_actual_forecast_{DATA_YM}.xlsx`. `DATA_YM` 한 곳만 바꾸면 월간 갱신. 모형·검증은 v2와 동일.

**v1 → v2 변경점**: OPM 직접예측(atanh) → **GPM(atanh 모형) − opex비율(달러 트렌드 모형) 분해**
1. GPM 모형에 **수출단가 QoQ** 항 추가 — 급등 국면 과소예측 편의 해소 (OOS 급등4분기 MAE 9.88 → 6.60%p)
2. **OPM < GPM 정합성 구조적 보장** (v1은 OPM 예측이 직전 GPM을 초과하는 비정합 발생 가능)
3. opex 달러는 관성이 강해 damped QoQ 트렌드로 외삽 → 매출 급증기 비율 축소가 기계적으로 반영

**예측 체계**: 매출 = f(수출액YoY, 난야YoY 선행) / GPM = f(수출액YoY, 단가QoQ, GPM관성) / opex$ = 트렌드 → OI = 매출 x (GPM − opex$/매출)

In [1]:
# ============ Cell 2: Control Panel ============
import os

DATA_YM = "202608"   # 데이터 기준월 - 다음 달 갱신 시 이 값만 변경 (예: "202609")

def _find(cands):
    """존재하는 첫 경로 반환 (데스크탑/노트북/클로드 컨테이너 자동 탐지)"""
    for p in cands:
        if os.path.exists(p):
            return p
    raise FileNotFoundError("파일 없음 - Control Panel 경로 확인:\n  " + "\n  ".join(cands))

# 854232 수출 데이터 (한국_월간수출분석 - 기준월 폴더 구조)
F_EXPORT = _find([
    rf"C:\Users\82108\OneDrive\INVESTMENT\한국주식\한국_월간수출분석\{DATA_YM}\854232_price_data.csv",
    rf"C:\Users\Hoyoung_Park\OneDrive\INVESTMENT\한국주식\한국_월간수출분석\{DATA_YM}\854232_price_data.csv",
    "/mnt/user-data/uploads/854232_price_data.csv",
])

# 대만기업 월별 매출 (지표상회_복제 - 파일명에 기준월 포함)
F_TW = _find([
    rf"C:\Users\82108\OneDrive\INVESTMENT\지표상회_복제\대만기업_월별매출분석\tw_actual_forecast_{DATA_YM}.xlsx",
    rf"C:\Users\Hoyoung_Park\OneDrive\INVESTMENT\지표상회_복제\대만기업_월별매출분석\tw_actual_forecast_{DATA_YM}.xlsx",
    f"/mnt/user-data/uploads/tw_actual_forecast_{DATA_YM}.xlsx",
])

# MU 분기 재무 (미국주식\raw_data)
F_MU = _find([
    r"C:\Users\82108\OneDrive\INVESTMENT\미국주식\raw_data\MU_fs_data.csv",
    r"C:\Users\Hoyoung_Park\OneDrive\INVESTMENT\미국주식\raw_data\MU_fs_data.csv",
    "/mnt/user-data/uploads/MU_fs_data.csv",
])

NANYA_ID, WINBOND_ID = 2408, 2344
OOS_START  = "2023-08-01"       # OOS 검증 시작
SUB_START  = "2020-08-31"       # 대만 포함 부표본 시작
OPEX_DAMP, OPEX_LOOK = 0.7, 4   # opex 트렌드 감쇠율 / 참조 분기수
OOS_MAPE_REV = 0.11             # 매출 OOS MAPE (밴드)
OOS_MAE_OPM  = 0.045            # v2 OPM OOS MAE (밴드, %p -> 소수)
print(f"F_EXPORT = {F_EXPORT}")
print(f"F_TW     = {F_TW}")
print(f"F_MU     = {F_MU}")

F_EXPORT = C:\Users\82108\OneDrive\INVESTMENT\한국주식\한국_월간수출분석\202608\854232_price_data.csv
F_TW     = C:\Users\82108\OneDrive\INVESTMENT\지표상회_복제\대만기업_월별매출분석\tw_actual_forecast_202608.xlsx
F_MU     = C:\Users\82108\OneDrive\INVESTMENT\미국주식\raw_data\MU_fs_data.csv


In [2]:
# ============ Cell 3: MU 분기 재무 로드 + 회계분기 매핑 ============
import pandas as pd, numpy as np
import statsmodels.api as sm

mu = pd.read_csv(F_MU, parse_dates=["date"])

def nearest_anchor(d):
    """MU 회계분기말(2/5/8/11월 말 부근 목요일) -> 가장 가까운 앵커 월말로 매핑"""
    cands = [pd.Timestamp(y, m, 1) + pd.offsets.MonthEnd(0)
             for y in (d.year - 1, d.year, d.year + 1) for m in (2, 5, 8, 11)]
    return min(cands, key=lambda a: abs((d - a).days))

mu["fq"] = mu["date"].apply(nearest_anchor)
assert mu["fq"].is_unique
mu = mu.sort_values("fq").reset_index(drop=True)
mu["opm"] = mu["operatingIncome"] / mu["revenue"]
mu["gpm"] = mu["grossProfit"] / mu["revenue"]
mu["opex"] = mu["grossProfit"] - mu["operatingIncome"]     # SG&A+R&D 등 (전 분기 양수 확인됨)
mu["opex_ratio"] = mu["opex"] / mu["revenue"]
for c in ["revenue", "operatingIncome"]:
    mu[f"{c}_yoy"] = mu[c].pct_change(4) * 100
print(mu[["fq", "revenue", "gpm", "opex_ratio", "opm"]].tail(6).round(3))

           fq      revenue    gpm  opex_ratio    opm
74 2025-02-28   8053000000  0.368       0.148  0.220
75 2025-05-31   9301000000  0.377       0.144  0.233
76 2025-08-31  11315000000  0.447       0.115  0.332
77 2025-11-30  13643000000  0.560       0.111  0.450
78 2026-02-28  23860000000  0.744       0.068  0.676
79 2026-05-31  41456000000  0.846       0.042  0.804


In [3]:
# ============ Cell 4: 수출·대만 월별 -> MU 분기 합산 + 패널 ============
def month_to_fq(p):
    """12·1·2월->2월말, 3·4·5->5월말, 6·7·8->8월말, 9·10·11->11월말"""
    m, y = p.month, p.year
    if m in (12, 1, 2): ay, am = (y + 1, 2) if m == 12 else (y, 2)
    elif m in (3, 4, 5): ay, am = y, 5
    elif m in (6, 7, 8): ay, am = y, 8
    else:                ay, am = y, 11
    return pd.Timestamp(ay, am, 1) + pd.offsets.MonthEnd(0)

exp = pd.read_csv(F_EXPORT, parse_dates=["date"])
exp["ym"] = exp["date"].dt.to_period("M")
exp["fq"] = exp["ym"].apply(month_to_fq)
g = exp.groupby("fq")
expq = pd.DataFrame({"exp_value": g["export_value"].sum(),
                     "exp_weight": g["export_weight"].sum(), "n_m": g.size()})
expq["exp_price"] = expq["exp_value"] / expq["exp_weight"]
expq = expq[expq["n_m"] == 3]
for c in ["exp_value", "exp_price"]:
    expq[f"{c}_yoy"] = expq[c].pct_change(4) * 100
expq["exp_price_qoq"] = expq["exp_price"].pct_change() * 100   # v2 신규: 단가 QoQ

tw = pd.read_excel(F_TW)
tw = tw[tw["kind"] == "actual"].copy()
tw["ym"] = pd.to_datetime(tw["date"]).dt.to_period("M")        # 2026-08-01 = 2026년 8월 매출
tw["fq"] = tw["ym"].apply(month_to_fq)
twq = {}
for cid, name in [(NANYA_ID, "nanya"), (WINBOND_ID, "winbond")]:
    s = tw[tw["company_id"] == cid].groupby("fq").agg(rev=("revenue", "sum"), n=("revenue", "size"))
    twq[name] = s[s["n"] == 3]["rev"].rename(name)
twq = pd.concat(twq.values(), axis=1)
for c in ["nanya", "winbond"]:
    twq[f"{c}_yoy"] = twq[c].pct_change(4) * 100

panel = mu.set_index("fq").join(expq, how="left").join(twq, how="left")
panel["nanya_yoy_l1"] = panel["nanya_yoy"].shift(1)
# atanh 변환 (유계 (-1,1) 보장용)
panel["gpm_z"] = np.arctanh(panel["gpm"].clip(-0.95, 0.95)); panel["gpm_z_l1"] = panel["gpm_z"].shift(1)
print("패널:", panel.shape, "| 기간:", panel.index.min().date(), "->", panel.index.max().date())

패널: (80, 30) | 기간: 2006-08-31 -> 2026-05-31


In [4]:
# ============ Cell 5: 매출 모형 (v1과 동일 - A4/A5/A6 앙상블) ============
def ols(y, X, data, label):
    d = data[[y] + X].dropna()
    m = sm.OLS(d[y], sm.add_constant(d[X])).fit()
    print(f"--- {label} | n={int(m.nobs)}, adjR2={m.rsquared_adj:.3f} ---")
    print(pd.DataFrame({"coef": m.params, "t": m.tvalues}).round(3).to_string(), "\n")
    return m

sub = panel.loc[SUB_START:]
mA4 = ols("revenue_yoy", ["exp_value_yoy"], sub,                    "A4: 수출액 단독 (2020Q3~)")
mA5 = ols("revenue_yoy", ["exp_value_yoy", "nanya_yoy"], sub,       "A5: 수출액+난야(동시)")
mA6 = ols("revenue_yoy", ["exp_value_yoy", "nanya_yoy_l1"], sub,    "A6: 수출액+난야(1분기 선행)")

--- A4: 수출액 단독 (2020Q3~) | n=24, adjR2=0.958 ---
                coef       t
const          2.052   0.515
exp_value_yoy  1.330  22.785 

--- A5: 수출액+난야(동시) | n=24, adjR2=0.966 ---
                coef       t
const          3.625   1.009
exp_value_yoy  1.092  10.427
nanya_yoy      0.093   2.615 

--- A6: 수출액+난야(1분기 선행) | n=24, adjR2=0.973 ---
                coef      t
const          6.252  1.850
exp_value_yoy  0.968  8.892
nanya_yoy_l1   0.196  3.676 



In [5]:
# ============ Cell 6: v2 마진 모형 - GPM(atanh) + opex$ 트렌드 분해 ============
# (1) GPM: atanh(GPM) ~ 수출액YoY + 단가QoQ + atanh(GPM)L1
#     - 단가 QoQ 가 급등 국면 반응속도를 담당 (YoY 관성만으로는 과소예측)
dG = panel[["gpm_z", "exp_value_yoy", "exp_price_qoq", "gpm_z_l1"]].dropna()
mG = sm.OLS(dG["gpm_z"], sm.add_constant(dG[["exp_value_yoy", "exp_price_qoq", "gpm_z_l1"]])).fit()
print(f"[GPM 모형] n={int(mG.nobs)}, adjR2={mG.rsquared_adj:.3f}")
print(pd.DataFrame({"coef": mG.params, "t": mG.tvalues}).round(4).to_string())

# (2) opex 달러: damped QoQ 트렌드 (관성 강함 -> 최근 4분기 QoQ 중앙값 x 감쇠 0.7)
def opex_forecast(hist, damp=OPEX_DAMP, look=OPEX_LOOK):
    g = hist.pct_change().dropna().iloc[-look:].median()
    return hist.iloc[-1] * (1 + damp * g)

# (3) OPM 예측 = tanh(GPM_z 예측) - opex$예측/매출 -> OPM < GPM 구조적 보장
def predict_opm(train, x_row, revenue_hat):
    d = train[["gpm_z", "exp_value_yoy", "exp_price_qoq", "gpm_z_l1"]].dropna()
    m = sm.OLS(d["gpm_z"], sm.add_constant(d[["exp_value_yoy", "exp_price_qoq", "gpm_z_l1"]])).fit()
    z = float(m.params["const"] + sum(m.params[f] * x_row[f] for f in ["exp_value_yoy", "exp_price_qoq", "gpm_z_l1"]))
    gpm_hat = np.tanh(z)
    oratio  = opex_forecast(train["opex"]) / revenue_hat
    return gpm_hat, oratio, gpm_hat - oratio

[GPM 모형] n=73, adjR2=0.869
                 coef        t
const          0.0267   1.4306
exp_value_yoy  0.0009   2.6182
exp_price_qoq  0.0026   3.1379
gpm_z_l1       0.8391  14.3236


In [6]:
# ============ Cell 7: OOS 검증 - v1(직접 OPM) vs v2(GPM 분해) ============
panel["opm_z"] = np.arctanh(panel["opm"].clip(-0.95, 0.95)); panel["opm_z_l1"] = panel["opm_z"].shift(1)
test_idx = [t for t in panel.index if t >= pd.Timestamp(OOS_START)]
rows = []
for t in test_idx:
    tr = panel.loc[:t].iloc[:-1]
    # v1: 직접 atanh(OPM)
    d1 = tr[["opm_z", "exp_value_yoy", "opm_z_l1"]].dropna()
    m1 = sm.OLS(d1["opm_z"], sm.add_constant(d1[["exp_value_yoy", "opm_z_l1"]])).fit()
    z1 = float(m1.params["const"] + m1.params["exp_value_yoy"] * panel.loc[t, "exp_value_yoy"]
               + m1.params["opm_z_l1"] * panel.loc[t, "opm_z_l1"])
    # v2: GPM 분해 (OPM 모형 자체 비교 위해 매출은 실제값 고정)
    x = panel.loc[t, ["exp_value_yoy", "exp_price_qoq", "gpm_z_l1"]]
    gpm_h, orat, opm_h = predict_opm(tr, x, panel.loc[t, "revenue"])
    rows.append({"fq": t, "v1": np.tanh(z1), "v2": opm_h, "gpm_hat": gpm_h,
                 "actual_opm": panel.loc[t, "opm"], "actual_gpm": panel.loc[t, "gpm"]})
oos = pd.DataFrame(rows).set_index("fq")
print((oos * 100).round(1).to_string())
print("\nOPM MAE(%p):  [급등4분기 = 2025-08 ~ 2026-05]")
for c in ["v1", "v2"]:
    e = (oos[c] - oos["actual_opm"]).abs() * 100
    print(f"  {c}: 전체 {e.mean():.2f} | 급등4분기 {e.iloc[-4:].mean():.2f}")

              v1    v2  gpm_hat  actual_opm  actual_gpm
fq                                                     
2023-08-31 -37.5 -38.9    -11.7       -36.7       -10.8
2023-11-30 -24.4 -24.5     -2.6       -23.9        -0.7
2024-02-29   0.9  -4.7     14.0         3.3        18.5
2024-05-31  18.2  16.7     29.5        10.6        26.9
2024-08-31  22.4  17.7     32.1        19.6        35.3
2024-11-30  25.1  23.8     38.5        25.0        38.4
2025-02-28  19.5  17.6     32.5        22.0        36.8
2025-05-31  19.2  20.2     33.4        23.3        37.7
2025-08-31  21.6  26.3     38.5        33.2        44.7
2025-11-30  31.4  37.4     46.9        45.0        56.0
2026-02-28  54.6  58.2     64.8        67.6        74.4
2026-05-31  79.0  77.9     82.1        80.4        84.6

OPM MAE(%p):  [급등4분기 = 2025-08 ~ 2026-05]
  v1: 전체 5.03 | 급등4분기 9.88
  v2: 전체 4.49 | 급등4분기 6.60


In [7]:
# ============ Cell 8: FY 나우캐스트 (대상 분기 자동 도출) ============
# 대상 분기 = 패널 마지막 실적 분기 + 3개월 -> 분기 구성 월/전년동기 월/기저 분기 전부 자동
last_fq = panel.index[-1]
tgt_fq  = last_fq + pd.DateOffset(months=3) + pd.offsets.MonthEnd(0)
NOWCAST_FQ = tgt_fq.strftime("%Y-%m-%d")                    # Cell 9(컨센서스)에서 사용
tm  = tgt_fq.to_period("M")
cm_ = [tm - 2, tm - 1, tm]                                  # 대상 분기 구성 3개월
pm_ = [m - 12 for m in cm_]                                 # 전년 동기 3개월
qm_ = [m - 3 for m in cm_]                                  # 직전 분기 3개월 (단가 QoQ용)
print(f"나우캐스트 대상 분기: {tgt_fq.date()} (마지막 실적 {last_fq.date()} 기준 자동)")

c_ = exp[exp["ym"].isin(cm_)]; p_ = exp[exp["ym"].isin(pm_)]; q_ = exp[exp["ym"].isin(qm_)]
n_have = c_["ym"].nunique()
if n_have < 3:
    print(f"경고: 대상 분기 수출 데이터 {n_have}/3개월 - 분기 중간 나우캐스트 (정밀도 하락)")
ev_yoy = (c_["export_value"].sum() / p_["export_value"].sum() - 1) * 100
price_qoq = ((c_["export_value"].sum() / c_["export_weight"].sum())
             / (q_["export_value"].sum() / q_["export_weight"].sum()) - 1) * 100
nanya_now = (tw[(tw.company_id == NANYA_ID) & tw.ym.isin(cm_)].revenue.sum()
             / tw[(tw.company_id == NANYA_ID) & tw.ym.isin(pm_)].revenue.sum() - 1) * 100
nanya_l1 = panel["nanya_yoy"].iloc[-1]
print(f"X: 수출액YoY {ev_yoy:+.1f}% | 단가QoQ {price_qoq:+.1f}% | 난야YoY {nanya_now:+.1f}% | 난야선행 {nanya_l1:+.1f}%")

# 매출: A4/A5/A6 등가중 앙상블 (기저 = 전년동기 분기, 자동)
base_fq = (tm - 12).to_timestamp() + pd.offsets.MonthEnd(0)
rev_base = panel.loc[base_fq, "revenue"]; oi_base = panel.loc[base_fq, "operatingIncome"]
xn = {"exp_value_yoy": ev_yoy, "nanya_yoy": nanya_now, "nanya_yoy_l1": nanya_l1}
yoys = [float(m.params["const"] + sum(m.params[f] * xn[f] for f in fs))
        for m, fs in [(mA4, ["exp_value_yoy"]), (mA5, ["exp_value_yoy", "nanya_yoy"]),
                      (mA6, ["exp_value_yoy", "nanya_yoy_l1"])]]
rev_yoy_hat = np.mean(yoys)
rev_hat = rev_base * (1 + rev_yoy_hat / 100)

# 마진: GPM 분해 (v2 모형)
x_now = pd.Series({"exp_value_yoy": ev_yoy, "exp_price_qoq": price_qoq,
                   "gpm_z_l1": np.arctanh(panel["gpm"].iloc[-1])})
gpm_hat, oratio_hat, opm_hat = predict_opm(panel, x_now, rev_hat)
oi_hat = rev_hat * opm_hat
print(f"\n[나우캐스트 {NOWCAST_FQ[:7]}]")
print(f"  매출: ${rev_hat/1e9:.1f}B (YoY {rev_yoy_hat:+.0f}%, 밴드 ${rev_hat*(1-OOS_MAPE_REV)/1e9:.0f}~{rev_hat*(1+OOS_MAPE_REV)/1e9:.0f}B)")
print(f"  GPM {gpm_hat*100:.1f}% - opex비율 {oratio_hat*100:.1f}% = OPM {opm_hat*100:.1f}%"
      f" (밴드 {max(opm_hat-OOS_MAE_OPM,0)*100:.0f}~{min(opm_hat+OOS_MAE_OPM,gpm_hat)*100:.0f}%)")
print(f"  영업이익: ${oi_hat/1e9:.1f}B (YoY {(oi_hat/oi_base-1)*100:+.0f}%)")

나우캐스트 대상 분기: 2026-08-31 (마지막 실적 2026-05-31 기준 자동)
X: 수출액YoY +270.9% | 단가QoQ +32.1% | 난야YoY +628.6% | 난야선행 +674.9%

[나우캐스트 2026-08]
  매출: $53.6B (YoY +374%, 밴드 $48~59B)
  GPM 88.4% - opex비율 3.4% = OPM 85.0% (밴드 81~88%)
  영업이익: $45.6B (YoY +1114%)


In [8]:
# ============ Cell 9: FMP 컨센서스 수집 (필드명 자동 인식) ============
# FMP_api_latest_T0_yoy_screener_v5 의 요청 패턴(재시도/429 처리) 재사용
import requests, time

API_KEY = "hT0gAk87j9xZx4PlBApvBqfVL5IahvgV"
EST_URLS = [   # stable 우선, 실패 시 legacy v3 폴백
    ("stable", "https://financialmodelingprep.com/stable/analyst-estimates",
     {"symbol": "MU", "period": "quarter", "page": 0, "limit": 12, "apikey": API_KEY}),
    ("v3",     "https://financialmodelingprep.com/api/v3/analyst-estimates/MU",
     {"period": "quarter", "limit": 12, "apikey": API_KEY}),
]

def fetch_estimates():
    for tag, url, params in EST_URLS:
        for k in range(3):
            try:
                r = requests.get(url, params=params, timeout=30)
                if r.status_code == 429:
                    time.sleep(1.5 + k); continue
                if r.status_code in (401, 402, 403):
                    print(f"[{tag}] HTTP {r.status_code} - 플랜 미포함 또는 키 문제: {r.text[:150]}")
                    break
                r.raise_for_status()
                data = r.json()
                if isinstance(data, dict) and "Error Message" in data:
                    print(f"[{tag}] {data['Error Message'][:150]}"); break
                if isinstance(data, list) and data:
                    print(f"[{tag}] 성공 - 응답 필드명:", sorted(data[0].keys()))
                    return pd.DataFrame(data)
                break
            except requests.RequestException as e:
                if k == 2: print(f"[{tag}] 요청 실패: {e}")
                time.sleep(0.5 + 0.5 * k)
    return None

def pick_col(df, candidates):
    """stable/legacy 필드명 후보 중 실제 존재하는 첫 컬럼 반환"""
    return next((c for c in candidates if c in df.columns), None)

est_raw = fetch_estimates()
consensus = None
if est_raw is not None:
    col_rev  = pick_col(est_raw, ["revenueAvg", "estimatedRevenueAvg"])
    col_ebit = pick_col(est_raw, ["ebitAvg", "estimatedEbitAvg"])
    col_eps  = pick_col(est_raw, ["epsAvg", "estimatedEpsAvg"])
    col_nrev = pick_col(est_raw, ["numAnalystsRevenue", "numberAnalystEstimatedRevenue", "numAnalystEstimatedRevenue"])
    est_raw["date"] = pd.to_datetime(est_raw["date"])
    est_raw["fq"] = est_raw["date"].apply(nearest_anchor)     # MU 회계분기 앵커로 매핑
    tgt = pd.Timestamp(NOWCAST_FQ)
    hit = est_raw[est_raw["fq"] == tgt]
    if len(hit) and col_rev:
        row = hit.iloc[0]
        consensus = {
            "fq": tgt, "rev": float(row[col_rev]),
            "ebit": float(row[col_ebit]) if col_ebit and pd.notna(row[col_ebit]) else None,
            "eps": float(row[col_eps]) if col_eps else None,
            "n_analysts": int(row[col_nrev]) if col_nrev and pd.notna(row[col_nrev]) else None,
        }
        if consensus["ebit"] is not None:
            consensus["implied_opm"] = consensus["ebit"] / consensus["rev"]
        print(f"\n[컨센서스 {tgt.date()}] 매출 ${consensus['rev']/1e9:.1f}B"
              + (f" | EBIT ${consensus['ebit']/1e9:.1f}B (내재 OPM {consensus['implied_opm']*100:.1f}%)" if consensus.get("ebit") else " | EBIT 없음")
              + (f" | 애널리스트 {consensus['n_analysts']}명" if consensus.get("n_analysts") else ""))
    else:
        print(f"\n대상 분기({tgt.date()}) 추정치 없음 - est_raw[['date','fq']] 확인 필요")
        print(est_raw[["date", "fq"]].head(8).to_string())
else:
    print("컨센서스 수집 실패 - 네트워크/플랜 확인 (클로드 컨테이너에서는 FMP 접근 불가, 로컬 실행 필요)")

[stable] HTTP 402 - 플랜 미포함 또는 키 문제: Premium Query Parameter: 'Special Endpoint : This value set for 'period' is not available under your current subscription please visit our subscriptio
[v3] 성공 - 응답 필드명: ['date', 'estimatedEbitAvg', 'estimatedEbitHigh', 'estimatedEbitLow', 'estimatedEbitdaAvg', 'estimatedEbitdaHigh', 'estimatedEbitdaLow', 'estimatedEpsAvg', 'estimatedEpsHigh', 'estimatedEpsLow', 'estimatedNetIncomeAvg', 'estimatedNetIncomeHigh', 'estimatedNetIncomeLow', 'estimatedRevenueAvg', 'estimatedRevenueHigh', 'estimatedRevenueLow', 'estimatedSgaExpenseAvg', 'estimatedSgaExpenseHigh', 'estimatedSgaExpenseLow', 'numberAnalystEstimatedRevenue', 'numberAnalystsEstimatedEps', 'symbol']

[컨센서스 2026-08-31] 매출 $50.4B | EBIT $40.9B (내재 OPM 81.1%) | 애널리스트 21명


In [9]:
# ============ Cell 10: 매수/매도 신호 판정 (판정카드) ============
# 임계치 = OOS 검증 오차 (매출 MAPE ±11%, OPM MAE ±4.5%p) - 자의성 배제
if consensus is None:
    print("컨센서스 없음 - Cell 9 를 로컬(데스크탑)에서 실행 후 재시도")
else:
    rev_gap = (rev_hat - consensus["rev"]) / consensus["rev"]
    sig_rev = "GREEN" if rev_gap > OOS_MAPE_REV else ("RED" if rev_gap < -OOS_MAPE_REV else "NEUTRAL")
    print(f"[매출 축] 모형 ${rev_hat/1e9:.1f}B vs 컨센서스 ${consensus['rev']/1e9:.1f}B"
          f" -> 갭 {rev_gap*100:+.1f}% (임계 ±{OOS_MAPE_REV*100:.0f}%) -> {sig_rev}")

    if consensus.get("implied_opm") is not None:
        opm_gap = opm_hat - consensus["implied_opm"]
        sig_opm = "GREEN" if opm_gap > OOS_MAE_OPM else ("RED" if opm_gap < -OOS_MAE_OPM else "NEUTRAL")
        print(f"[OPM 축]  모형 {opm_hat*100:.1f}% vs 내재 {consensus['implied_opm']*100:.1f}%"
              f" -> 갭 {opm_gap*100:+.1f}%p (임계 ±{OOS_MAE_OPM*100:.1f}%p) -> {sig_opm}")
    else:
        sig_opm = "N/A"
        print("[OPM 축]  EBIT 컨센서스 없음 -> 매출 축 단독 판정 (신뢰도 하향)")

    verdict = ("매수 우위 (발표 전 진입)" if sig_rev == "GREEN" and sig_opm in ("GREEN", "N/A")
               else "매도/관망" if sig_rev == "RED"
               else "무포지션 (신호 혼재)")
    print(f"\n>>> 카드 판정: {verdict} <<<")
    print(">>> 카드가 판정했으므로 그대로 실행 - 발표일까지 재량 개입 없음")

[매출 축] 모형 $53.6B vs 컨센서스 $50.4B -> 갭 +6.3% (임계 ±11%) -> NEUTRAL
[OPM 축]  모형 85.0% vs 내재 81.1% -> 갭 +3.9%p (임계 ±4.5%p) -> NEUTRAL

>>> 카드 판정: 무포지션 (신호 혼재) <<<
>>> 카드가 판정했으므로 그대로 실행 - 발표일까지 재량 개입 없음


In [10]:
# ============ Cell 11: 컨센서스 vintage 스냅샷 적재 (백테스트용 축적) ============
# FMP 는 현재 스냅샷만 제공 -> 조회 시점(as_of)별로 쌓아야 look-ahead 없는 백테스트 가능
if consensus is not None:
    snap_path = os.path.join(os.path.dirname(F_MU), "MU_consensus_vintage.csv")
    snap = pd.DataFrame([{
        "as_of": pd.Timestamp.today().normalize(), "fq": consensus["fq"],
        "consensus_rev": consensus["rev"], "consensus_ebit": consensus.get("ebit"),
        "consensus_eps": consensus.get("eps"), "n_analysts": consensus.get("n_analysts"),
        "model_rev": rev_hat, "model_opm": opm_hat, "model_oi": oi_hat,
    }])
    header = not os.path.exists(snap_path)
    snap.to_csv(snap_path, mode="a", header=header, index=False, encoding="utf-8-sig")
    print(f"vintage 적재: {snap_path} (header={header})")
    # 추후 MySQL(investar) 이관 시: created_at 고정 + updated_at 갱신 UPSERT 패턴 사용

vintage 적재: C:\Users\82108\OneDrive\INVESTMENT\미국주식\raw_data\MU_consensus_vintage.csv (header=True)


In [11]:
# ============ Cell 12: 난야 실적 국면 판별 (월간 타이밍 신호) ============
# 규칙 (MU_price_signal_backtest_v2 에서 채택): 난야 3M-MA YoY 의 ΔYoY > 0 -> 보유 / <= 0 -> 현금
# 데이터 공표: 매월 ~10일 (전월 대만 매출) -> 판정 포지션은 11일 ~ 익월 10일 적용
nanya_m = tw[tw["company_id"] == NANYA_ID].set_index("ym")["revenue"].sort_index()
n_yoy   = nanya_m.pct_change(12) * 100
n_ma3   = n_yoy.rolling(3).mean()
n_delta = n_ma3.diff()

regime_tbl = pd.DataFrame({
    "매출(십억TWD)": nanya_m / 1e6, "YoY(%)": n_yoy, "3M-MA(%)": n_ma3, "dMA(%p)": n_delta,
    "판정": np.where(n_delta > 0, "보유", np.where(n_delta.notna(), "현금", "-")),
}).tail(6)
print("난야 국면 신호 (최근 6개월):")
print(regime_tbl.round(1).to_string())

REGIME = "보유" if n_delta.iloc[-1] > 0 else "현금"
_asof_m = n_delta.index[-1]
print(f"\n>>> 현재 국면({_asof_m} 데이터): {REGIME}  (dMA {n_delta.iloc[-1]:+.1f}%p)")
print(f">>> 적용 구간: {(_asof_m+1).to_timestamp().strftime('%Y-%m')}-11 ~ {(_asof_m+2).to_timestamp().strftime('%Y-%m')}-10")
print("(참고) 원신호 ΔYoY:", f"{n_yoy.diff().iloc[-1]:+.1f}%p", "- 병행 관찰용, 판정은 평활 기준")

난야 국면 신호 (최근 6개월):
         매출(십억TWD)  YoY(%)  3M-MA(%)  dMA(%p)  판정
ym                                               
2026-03       18.2   560.0     584.9     38.4  보유
2026-04       25.5   717.3     621.4     36.4  보유
2026-05       27.7   730.1     669.2     47.8  보유
2026-06       29.4   621.3     689.6     20.4  보유
2026-07       43.9   719.6     690.4      0.8  보유
2026-08       44.7   560.9     633.9    -56.4  현금

>>> 현재 국면(2026-08 데이터): 현금  (dMA -56.4%p)
>>> 적용 구간: 2026-09-11 ~ 2026-10-10
(참고) 원신호 ΔYoY: -158.8%p - 병행 관찰용, 판정은 평활 기준


In [12]:
# ============ Cell 13: 종합 판정 (분기 카드 x 월간 국면) ============
# 사전확약 규칙 (초안):
#   평상시                    -> 월간 국면 신호(REGIME)대로 보유/현금
#   실적발표 이벤트 윈도우      -> 분기 카드 판정 우선 (나우캐스트 vs 컨센서스)
#     (윈도우 = 대만 분기 마지막 달 발표일 ~ MU 실적발표일, 약 2~3주)
#   카드 GREEN 인데 국면 현금  -> 이벤트 한정 보유 후 발표일에 국면 신호로 복귀
print("=" * 60)
print("종합 판정")
print("=" * 60)
print(f"[월간 국면]  {REGIME}  (난야 3M-MA dYoY 기준)")
if consensus is not None:
    print(f"[분기 카드]  {verdict}  (나우캐스트 vs 컨센서스)")
    if "매수" in verdict and REGIME == "현금":
        final = "이벤트 한정 보유 (발표일 이후 국면 신호로 복귀)"
    elif "매도" in verdict or "무포지션" in verdict:
        final = "무포지션 (카드 우선)"
    else:
        final = "보유" if REGIME == "보유" else "이벤트 구간 보유, 평상시 현금"
    print(f"\n>>> 최종: {final}")
else:
    print("[분기 카드]  미판정 (컨센서스 미수집 - 로컬에서 Cell 9 실행 필요)")
    print(f"\n>>> 평상시 규칙 적용: {REGIME}")
print(">>> 카드/신호가 판정했으므로 그대로 실행 - 재량 개입 없음")

종합 판정
[월간 국면]  현금  (난야 3M-MA dYoY 기준)
[분기 카드]  무포지션 (신호 혼재)  (나우캐스트 vs 컨센서스)

>>> 최종: 무포지션 (카드 우선)
>>> 카드/신호가 판정했으므로 그대로 실행 - 재량 개입 없음


## 컨센서스 신호 사용 시 주의
- Cell 9~11 은 FMP 실시간 호출이라 **데스크탑/노트북에서만 실행** 가능 (클로드 컨테이너는 FMP 차단)
- 첫 실행 시 `응답 필드명` 출력을 확인 — stable/legacy 어느 쪽이든 자동 매핑되지만, 둘 다 아닌 새 이름이면 `pick_col` 후보에 추가
- EBIT 추정치가 플랜에 없으면 매출 축 단독 판정으로 자동 강등 (판정 신뢰도 하향으로 해석)
- vintage CSV 는 조회할 때마다 1행씩 쌓임 — 분기당 최소 발표 2~3주 전, 1주 전, 전일 3회 수집 권장
- Cell 12(국면 신호)는 대만 데이터만 있으면 어디서든 실행 가능. Cell 13 종합 판정의 이벤트 오버라이드 규칙은 초안이며, 확정 전 판정카드 문서에 명문화할 것

## 매수/매도 판정 프레임 (다음 단계 - 판정카드 연동)
나우캐스트가 완성되는 시점(대만 8월 실적 공개 ~9/10)부터 MU 발표(~9월 말)까지가 실행 윈도우.

| 축 | 신호 | GREEN (매수 우위) | RED (매도/관망) |
|---|---|---|---|
| 매출 | (나우캐스트 − 컨센서스)/컨센서스 | > +11% (OOS MAPE 초과 괴리) | < −11% |
| OPM | 나우캐스트 − 컨센서스 내재 OPM | > +4.5%p | < −4.5%p |

- 컨센서스는 FMP `analyst-estimates` 엔드포인트로 수집 예정 (v3 과제)
- 2축 모두 GREEN이면 발표 전 진입, 혼재 시 카드 규칙에 따라 무포지션 — "카드가 판정했으므로 그대로 실행"
- **한계 유지**: 현 X값이 학습범위 상단 부근 외삽 / 급등 국면 잔여 과소편의(v2에도 소폭 존재) → 상방 서프라이즈 방향 오차 가능성이 하방보다 큼